# VisClick — Phase 4.2 / D-01 (step 1 of 2): train source-domain DETR-R50

**Prerequisites:**
- `05_train_source.ipynb` must already have been run **once** in any prior session, so that `<DRIVE>/data/source_train_bundles/{train,val,test}.tar.gz` exist on Drive. This notebook **reuses** the same CLAY data the YOLOv8 source baseline was trained on, so the DETR-vs-YOLOv8 comparison is apples-to-apples.

**Companion to:** `05_train_source.ipynb` (this notebook's YOLOv8 counterpart) and `10_detr_finetune.ipynb` (next step).

**This notebook** (Phase 4.2 — D-01 — DETR-R50 backbone):
1. Mount Drive → `git pull` → GPU check → install `transformers` + `timm` + `pycocotools` + `torchmetrics`.
2. **Bootstrap `/content/source_train`** from the same Drive tar bundles `05_train_source.ipynb` writes (instant on a fresh Colab runtime).
3. **Convert** the YOLO-format labels to **COCO JSON** (DETR's native format). One JSON per split, written to `/content/source_train_coco/`.
4. **Train DETR-R50** from `facebook/detr-resnet-50` (HF pretrained on COCO) on the CLAY UI corpus. Persisted to `<DRIVE>/weights/baseline_source_detr/run1/` so it survives runtime restarts.
5. **Resume-aware** like 05: existing `best.pt` → skip, `last.pt` → resume, else fresh.
6. **Save stable name** `baseline_source_detr/best_source_detr_r50.pt` for `10_detr_finetune.ipynb`.
7. **Eval** mAP@0.5 + mAP@0.5:0.95 on the source test split via `torchmetrics.detection.MeanAveragePrecision`.
8. Write metrics CSV `<DRIVE>/reports/tables/source_domain_results_detr.csv` (matches the schema of `source_domain_results.csv` so they merge into a single T-01 row).

**Compute reality check.** DETR-R50 at imgsz=800 batch=2 (T4's realistic limit) is ~3-4× slower per step than YOLOv8s at imgsz=640 batch=16. To stay inside Colab Free's session limits, the default schedule is **8 epochs** (vs YOLOv8's 30). DETR converges faster on COCO-like detection so this is reasonable; if your run finishes early bump `EPOCHS` to 12.

**Report.** Every step prints `REPORT ...` lines for the data form §4 row *Source baseline (DETR-R50)*.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import sys, subprocess
# Pin Pillow below 12.0.0: that release ships an internal import bug
# (PIL.ImageText references PIL._typing._Ink which the same release does
# not define). The bug breaks every downstream torchvision import that
# touches the dataset pipeline. 11.3.0 is the last known-good version
# on Colab Python 3.12 as of 26 May 2026.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "Pillow==11.3.0",
     "transformers", "timm", "accelerate", "pycocotools", "torchmetrics"],
    check=False,
)
import torch, transformers, PIL
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("transformers:", transformers.__version__, "| Pillow:", PIL.__version__)
print("REPORT env | torch =", torch.__version__,
      "| cuda =", torch.cuda.is_available(),
      "| transformers =", transformers.__version__,
      "| pillow =", PIL.__version__)

## 9.1 — Bootstrap `/content/source_train` from Drive bundles

Same logic as `05_train_source.ipynb` 5.1 — extracts the three tarballs that `04_assemble_source.ipynb` produced, recreates symlinks into `unified/<split>/images/`, writes a `data.yaml`. Idempotent.

If the bundles do not exist on Drive, run `04_assemble_source.ipynb` once first.

In [ ]:
import os, tarfile, time
DRIVE   = "/content/drive/MyDrive/visclick"
UNIFIED = os.path.join(DRIVE, "data", "unified")
BUNDLES = os.path.join(DRIVE, "data", "source_train_bundles")
SRC = "/content/source_train"
DATA_YAML = os.path.join(SRC, "data.yaml")
SPLITS = ["train", "val", "test"]
CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]

def _count(p):
    try:
        n = 0
        with os.scandir(p) as it:
            for e in it:
                if e.is_file(follow_symlinks=False) or e.is_symlink():
                    n += 1
        return n
    except OSError:
        return 0

def _bootstrap_from_bundles():
    os.makedirs(SRC, exist_ok=True)
    for sp in SPLITS:
        b = os.path.join(BUNDLES, f"{sp}.tar.gz")
        if not (os.path.isfile(b) and os.path.getsize(b) > 1000):
            return False, f"bundle missing for {sp}: {b}"
    for sp in SPLITS:
        t0 = time.time()
        with tarfile.open(os.path.join(BUNDLES, f"{sp}.tar.gz"), "r:gz") as tf:
            tf.extractall(SRC)
        manifest = os.path.join(SRC, "manifests", f"{sp}.txt")
        if not os.path.isfile(manifest):
            return False, f"manifest missing in bundle for {sp}"
        with open(manifest) as fh:
            names = [ln.strip() for ln in fh if ln.strip()]
        src_img_dir = os.path.join(UNIFIED, sp, "images")
        dst_img_dir = os.path.join(SRC, "images", sp)
        os.makedirs(dst_img_dir, exist_ok=True)
        made = 0
        for fn in names:
            dst = os.path.join(dst_img_dir, fn)
            if os.path.islink(dst) or os.path.isfile(dst):
                continue
            try:
                os.symlink(os.path.join(src_img_dir, fn), dst); made += 1
            except OSError:
                pass
        print(f"bootstrap {sp}: labels + {made} symlinks (manifest {len(names)}) in {time.time()-t0:0.1f}s")
    if not os.path.isfile(DATA_YAML):
        import yaml as _yaml
        with open(DATA_YAML, "w") as fh:
            _yaml.safe_dump({"path": SRC, "train": "images/train", "val": "images/val",
                              "test": "images/test", "nc": len(CLASSES), "names": CLASSES},
                             fh, sort_keys=False)
        print("wrote", DATA_YAML)
    return True, "OK"

if not os.path.isfile(DATA_YAML):
    print("source_train missing locally; bootstrapping from Drive bundles...")
    ok, msg = _bootstrap_from_bundles()
    print("REPORT step = BOOTSTRAP | status =", "RESTORED" if ok else "FAILED", "| msg =", msg)

assert os.path.isfile(DATA_YAML), f"Missing {DATA_YAML}. Run 04_assemble_source.ipynb first to create Drive bundles."
for sp in ("train", "val", "test"):
    img_n = _count(os.path.join(SRC, "images", sp))
    lbl_n = _count(os.path.join(SRC, "labels", sp))
    print(f"REPORT precheck | split = {sp:5s} | images = {img_n:>5d} | labels = {lbl_n:>5d}")
    assert img_n > 0 and lbl_n > 0, f"{sp}: empty images/labels — re-run 04"

## 9.2 — Convert YOLO labels to COCO JSON

DETR ingests data via HuggingFace's `CocoDetection`-style format: one JSON per split listing images, categories, and per-box annotations. We convert in-place once per split and cache the result under `/content/source_train_coco/{train,val,test}.json`. Idempotent — skips if the JSON already exists and is newer than the last label file.

Class IDs are kept identical to the YOLO mapping so a side-by-side comparison with `05_train_source.ipynb` is straightforward: `0=button, 1=text, 2=text_input, 3=icon, 4=menu, 5=checkbox`.

In [ ]:
import os, json, time
from PIL import Image, UnidentifiedImageError

COCO_ROOT = "/content/source_train_coco"
os.makedirs(COCO_ROOT, exist_ok=True)

CATEGORIES = [{"id": i, "name": n, "supercategory": "ui"} for i, n in enumerate(CLASSES)]

# Tolerant image probe: returns (w, h) or None if Drive FUSE can't read the file.
# RR-08 / RR-09 in the risk register: Drive FUSE I/O instability on dirs with
# 10k+ files. We treat any OSError, UnidentifiedImageError, or zero-byte file as
# "skip this image," log the count, and keep going.
def _probe_image(img_path: str, retries: int = 2):
    for attempt in range(retries + 1):
        try:
            if not os.path.isfile(img_path):
                return None
            if os.path.getsize(img_path) == 0:
                return None
            with Image.open(img_path) as im:
                im.verify()  # raises on truncated / corrupt files
            # verify() leaves the file in an unloadable state; reopen for size.
            with Image.open(img_path) as im:
                return im.size  # (w, h)
        except (OSError, UnidentifiedImageError):
            if attempt < retries:
                time.sleep(0.2 * (attempt + 1))
                continue
            return None
    return None

def _yolo_to_coco_split(split: str) -> str:
    img_dir = os.path.join(SRC, "images", split)
    lbl_dir = os.path.join(SRC, "labels", split)
    out_json = os.path.join(COCO_ROOT, f"{split}.json")
    # Skip if up-to-date.
    if os.path.isfile(out_json):
        try:
            lbl_mtime = max(
                (os.path.getmtime(os.path.join(lbl_dir, f))
                 for f in os.listdir(lbl_dir) if f.endswith(".txt")),
                default=0,
            )
            if os.path.getmtime(out_json) >= lbl_mtime:
                with open(out_json) as fh:
                    j = json.load(fh)
                print(f"REPORT yolo2coco | split = {split:5s} | status = CACHED "
                      f"| images = {len(j['images'])} | annotations = {len(j['annotations'])}")
                return out_json
        except OSError:
            pass

    images, annotations = [], []
    img_id, ann_id = 0, 0
    img_files = sorted(
        f for f in os.listdir(img_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    t0 = time.time()
    skipped, skipped_examples = 0, []
    for fn in img_files:
        stem = os.path.splitext(fn)[0]
        img_path = os.path.join(img_dir, fn)
        size = _probe_image(img_path)
        if size is None:
            skipped += 1
            if len(skipped_examples) < 5:
                skipped_examples.append(fn)
            continue
        w, h = size
        images.append({
            "id": img_id,
            "file_name": fn,
            "width": w,
            "height": h,
        })
        lbl_path = os.path.join(lbl_dir, stem + ".txt")
        if os.path.isfile(lbl_path):
            with open(lbl_path) as fh:
                for line in fh:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls = int(parts[0])
                    cx, cy, bw, bh = (float(p) for p in parts[1:])
                    x = max(0.0, (cx - bw / 2.0) * w)
                    y = max(0.0, (cy - bh / 2.0) * h)
                    bw_abs = min(w - x, bw * w)
                    bh_abs = min(h - y, bh * h)
                    if bw_abs <= 1 or bh_abs <= 1:
                        continue
                    annotations.append({
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": cls,
                        "bbox": [x, y, bw_abs, bh_abs],
                        "area": bw_abs * bh_abs,
                        "iscrowd": 0,
                    })
                    ann_id += 1
        img_id += 1

    coco = {
        "info": {"description": f"VisClick source_train ({split}), YOLO->COCO converted"},
        "images": images,
        "annotations": annotations,
        "categories": CATEGORIES,
    }
    with open(out_json, "w") as fh:
        json.dump(coco, fh)
    skipped_note = f" | skipped = {skipped}" if skipped else ""
    if skipped:
        examples = ", ".join(skipped_examples)
        more = f" (+{skipped - len(skipped_examples)} more)" if skipped > len(skipped_examples) else ""
        skipped_note += f" | examples = [{examples}{more}]"
    print(f"REPORT yolo2coco | split = {split:5s} | status = WROTE "
          f"| images = {len(images)} | annotations = {len(annotations)}"
          f"{skipped_note} | elapsed_s = {time.time()-t0:0.1f}")
    return out_json

train_json = _yolo_to_coco_split("train")
val_json   = _yolo_to_coco_split("val")
test_json  = _yolo_to_coco_split("test")
print("COCO JSONs:", train_json, val_json, test_json)

## 9.3 — Build DETR dataset + processor

`DetrImageProcessor` handles both image normalisation and ground-truth boxes. We wrap `torchvision.datasets.CocoDetection` so the processor's `__call__` returns model-ready tensors. A collate function pads variable-size images for batching on the T4.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection
from transformers import DetrImageProcessor, DetrForObjectDetection

HF_MODEL = "facebook/detr-resnet-50"
# Cap the input resolution to keep training inside a single Colab Free session.
# Default DetrImageProcessor uses {shortest_edge: 800, longest_edge: 1333}, which
# at batch=2 takes ~2 hr/epoch on T4. With {600, 800} it drops to ~50 min/epoch.
# Trade-off: smaller text/icon recall on the source set; matches our target use.
processor = DetrImageProcessor.from_pretrained(
    HF_MODEL,
    size={"shortest_edge": 600, "longest_edge": 800},
)

class VisClickCocoDetection(CocoDetection):
    """Wrap CocoDetection so the DetrImageProcessor encodes (image, anns) once."""
    def __init__(self, img_folder, ann_file, processor):
        super().__init__(img_folder, ann_file)
        self.processor = processor

    def __getitem__(self, idx):
        img, target = super().__getitem__(idx)
        image_id = self.ids[idx]
        target = {"image_id": image_id, "annotations": target}
        encoding = self.processor(images=img, annotations=target, return_tensors="pt")
        # processor returns batched tensors; squeeze.
        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]
        return pixel_values, labels

def collate_fn(batch):
    # Manual variable-size padding. We avoid processor.pad(...) because the
    # `return_tensors` kwarg was dropped from DetrImageProcessor.pad in recent
    # transformers releases. Inputs are already normalised tensors from
    # __getitem__, so we just pad spatially and build a pixel mask.
    pixel_values = [b[0] for b in batch]
    labels = [b[1] for b in batch]

    max_h = max(pv.shape[1] for pv in pixel_values)
    max_w = max(pv.shape[2] for pv in pixel_values)
    bsz = len(pixel_values)

    padded_pv = torch.zeros(bsz, 3, max_h, max_w, dtype=pixel_values[0].dtype)
    pixel_mask = torch.zeros(bsz, max_h, max_w, dtype=torch.long)
    for i, pv in enumerate(pixel_values):
        _, h, w = pv.shape
        padded_pv[i, :, :h, :w] = pv
        pixel_mask[i, :h, :w] = 1

    return {
        "pixel_values": padded_pv,
        "pixel_mask": pixel_mask,
        "labels": labels,
    }

train_ds = VisClickCocoDetection(os.path.join(SRC, "images", "train"), train_json, processor)
val_ds   = VisClickCocoDetection(os.path.join(SRC, "images", "val"),   val_json,   processor)
test_ds  = VisClickCocoDetection(os.path.join(SRC, "images", "test"),  test_json,  processor)
print(f"REPORT dataset | train = {len(train_ds)} | val = {len(val_ds)} | test = {len(test_ds)}")

# Sanity: one sample shape.
px, lab = train_ds[0]
print("sample pixel_values:", tuple(px.shape), "| labels keys:", list(lab.keys()))

## 9.4 — Train DETR (fresh / resume / skip)

**Resume logic.**
- `FORCE_FRESH=True` (default) → ignore prior checkpoints, train from `facebook/detr-resnet-50`.
- `FORCE_FRESH=False` + `last.pt` with `last_epoch+1 >= EPOCHS` → training is complete, load `best.pt`.
- `FORCE_FRESH=False` + `last.pt` partial → resume from `last_epoch + 1`.
- No checkpoint → fresh.

**Colab Free fit.** The previous 8-epoch / imgsz=1333 / fp32 config took ~2 hr/epoch on T4 and didn't fit one session. This config uses:
- `EPOCHS = 6` (down from 8)
- `processor.size = {shortest: 600, longest: 800}` (cuts pixel count ~2.6x)
- `USE_AMP = True` (fp16 autocast + GradScaler, ~30% speedup on T4)
Expected: ~50 min/epoch × 6 = ~5 hr total, fits one session with margin.

**Hyperparameters** unchanged from the HF reference recipe: AdamW, LR 1e-4 for new heads / 1e-5 for backbone, weight decay 1e-4, gradient accumulation 8 (effective batch 16 at micro-batch 2). Step-level progress prints every `LOG_EVERY=100` micro-batches so you can monitor the loss curve.

In [ ]:
import os, json, time, math
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

PROJECT = os.path.join(DRIVE, "weights", "baseline_source_detr")
NAME    = "run1"
RUN_DIR = os.path.join(PROJECT, NAME)
WTS_DIR = os.path.join(RUN_DIR, "weights")
LAST_PT = os.path.join(WTS_DIR, "last.pt")
BEST_PT = os.path.join(WTS_DIR, "best.pt")
META_JS = os.path.join(WTS_DIR, "meta.json")
os.makedirs(WTS_DIR, exist_ok=True)

# FORCE_FRESH = True wipes the previous checkpoints' INFLUENCE: the cell still
# saves into the same run dir, but starts from facebook/detr-resnet-50 weights.
# Set to False if you want to resume from the last completed epoch (last.pt).
FORCE_FRESH = True
EPOCHS      = 6       # 6 epochs at imgsz=800 fp16 ~ 4-5 hr on T4 (fits Colab Free)
MICRO_BATCH = 2       # T4 friendly
ACCUM       = 8       # effective batch 16
LR_HEAD     = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY= 1e-4
NUM_WORKERS = 2
USE_AMP     = True    # fp16 mixed precision (T4 supports it natively, ~30% speedup)
LOG_EVERY   = 100     # print step-level REPORT lines every N micro-batches

device = "cuda" if torch.cuda.is_available() else "cpu"

def _build_model(from_path: str | None) -> DetrForObjectDetection:
    if from_path and os.path.isfile(from_path):
        print("loading weights from", from_path)
        model = DetrForObjectDetection.from_pretrained(
            HF_MODEL,
            num_labels=len(CLASSES),
            ignore_mismatched_sizes=True,
            id2label={i: n for i, n in enumerate(CLASSES)},
            label2id={n: i for i, n in enumerate(CLASSES)},
        )
        state = torch.load(from_path, map_location="cpu")
        model.load_state_dict(state["model"], strict=False)
        return model
    return DetrForObjectDetection.from_pretrained(
        HF_MODEL,
        num_labels=len(CLASSES),
        ignore_mismatched_sizes=True,
        id2label={i: n for i, n in enumerate(CLASSES)},
        label2id={n: i for i, n in enumerate(CLASSES)},
    )

def _save_ckpt(model: DetrForObjectDetection, path: str, meta: dict) -> None:
    torch.save({"model": model.state_dict(), "meta": meta}, path)
    with open(META_JS, "w") as fh:
        json.dump(meta, fh)

def _build_optim(model: DetrForObjectDetection) -> AdamW:
    backbone_params, head_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in n:
            backbone_params.append(p)
        else:
            head_params.append(p)
    return AdamW(
        [
            {"params": head_params,     "lr": LR_HEAD},
            {"params": backbone_params, "lr": LR_BACKBONE},
        ],
        weight_decay=WEIGHT_DECAY,
    )

t0 = time.time()
train_status = None
start_epoch = 0

def _read_last_epoch() -> int:
    if not os.path.isfile(META_JS):
        return -1
    try:
        with open(META_JS) as fh:
            return int(json.load(fh).get("last_epoch", -1))
    except (json.JSONDecodeError, OSError):
        return -1

if FORCE_FRESH:
    print(f"FORCE_FRESH=True -> ignoring prior checkpoints; training fresh from {HF_MODEL}")
    model = _build_model(None).to(device)
    train_status = "FRESH"
elif os.path.isfile(LAST_PT) and _read_last_epoch() + 1 >= EPOCHS:
    print(f"last.pt at epoch {_read_last_epoch()} >= EPOCHS={EPOCHS} -> training complete, loading best.pt")
    src = BEST_PT if os.path.isfile(BEST_PT) else LAST_PT
    model = _build_model(src).to(device)
    train_status = "SKIP_ALREADY_TRAINED"
elif os.path.isfile(LAST_PT):
    start_epoch = _read_last_epoch() + 1
    print(f"last.pt found at epoch {start_epoch - 1} -> resuming from epoch {start_epoch}/{EPOCHS}: {LAST_PT}")
    model = _build_model(LAST_PT).to(device)
    train_status = "RESUMED"
else:
    print("no prior checkpoint -> training fresh from", HF_MODEL)
    model = _build_model(None).to(device)
    train_status = "FRESH"

if train_status != "SKIP_ALREADY_TRAINED":
    optim = _build_optim(model)
    train_loader = DataLoader(
        train_ds, batch_size=MICRO_BATCH, shuffle=True,
        collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=MICRO_BATCH, shuffle=False,
        collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True,
    )

    best_val_loss = float("inf")
    if not FORCE_FRESH and os.path.isfile(META_JS):
        try:
            with open(META_JS) as fh:
                best_val_loss = float(json.load(fh).get("best_val_loss", float("inf")))
        except (json.JSONDecodeError, OSError):
            pass

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    amp_dtype = torch.float16
    steps_per_epoch = len(train_loader)
    print(f"REPORT cfg | epochs = {EPOCHS} | start_epoch = {start_epoch} "
          f"| micro_batch = {MICRO_BATCH} | accum = {ACCUM} "
          f"| amp = {USE_AMP} | steps_per_epoch = {steps_per_epoch} "
          f"| imgsz_cap = {processor.size}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        ep_loss, ep_steps = 0.0, 0
        epoch_t0 = time.time()
        optim.zero_grad()
        for step, batch in enumerate(train_loader):
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            pixel_mask = batch["pixel_mask"].to(device, non_blocking=True)
            labels = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in batch["labels"]]
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=USE_AMP):
                out = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
                loss = out.loss / ACCUM
            scaler.scale(loss).backward()
            ep_loss += float(out.loss.detach())
            ep_steps += 1
            if (step + 1) % ACCUM == 0:
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
                scaler.step(optim)
                scaler.update()
                optim.zero_grad()
            if (step + 1) % LOG_EVERY == 0:
                running = ep_loss / max(1, ep_steps)
                pct = 100.0 * (step + 1) / steps_per_epoch
                elapsed_min = (time.time() - epoch_t0) / 60.0
                print(f"REPORT step | epoch = {epoch+1}/{EPOCHS} "
                      f"| step = {step+1:>5d}/{steps_per_epoch} ({pct:5.1f}%) "
                      f"| running_loss = {running:0.4f} "
                      f"| elapsed_min = {elapsed_min:0.1f}")
        # Flush trailing partial accumulation.
        if ep_steps % ACCUM != 0:
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            scaler.step(optim)
            scaler.update()
            optim.zero_grad()

        train_loss = ep_loss / max(1, ep_steps)

        # Validation loss (also under autocast for speed; no grad scaling needed).
        model.eval()
        val_loss, val_steps = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch["pixel_values"].to(device, non_blocking=True)
                pixel_mask = batch["pixel_mask"].to(device, non_blocking=True)
                labels = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in batch["labels"]]
                with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=USE_AMP):
                    out = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
                val_loss += float(out.loss.detach())
                val_steps += 1
        val_loss = val_loss / max(1, val_steps)

        # Persist last + (maybe) best.
        meta = {"last_epoch": epoch, "best_val_loss": best_val_loss}
        _save_ckpt(model, LAST_PT, meta)
        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            meta["best_val_loss"] = best_val_loss
            _save_ckpt(model, BEST_PT, meta)
        epoch_min = (time.time() - epoch_t0) / 60.0
        print(f"REPORT epoch | n = {epoch+1:>2d}/{EPOCHS} "
              f"| train_loss = {train_loss:0.4f} | val_loss = {val_loss:0.4f} "
              f"| best = {best_val_loss:0.4f} | improved = {improved} "
              f"| epoch_min = {epoch_min:0.1f}")

elapsed = time.time() - t0
print(f"REPORT step = TRAIN | status = {train_status} | elapsed_s = {elapsed:0.0f} | run_dir = {RUN_DIR}")

## 9.5 — Save stable name

Later notebooks reference `baseline_source_detr/best_source_detr_r50.pt` without needing to know the run number. Same convention as YOLOv8's `best_source_v8s.pt`.

In [ ]:
import shutil
STABLE = os.path.join(PROJECT, "best_source_detr_r50.pt")
if os.path.isfile(BEST_PT):
    shutil.copy2(BEST_PT, STABLE)
    print("REPORT step = SAVE_STABLE | src =", BEST_PT, "| dst =", STABLE,
          "| bytes =", os.path.getsize(STABLE))
else:
    print("NOTE: best.pt missing; training may not have produced a best checkpoint yet.")
    print("REPORT step = SAVE_STABLE | status = MISSING_BEST")

## 9.6 — Evaluate on source test split (mAP@0.5 + mAP@0.5:0.95)

Uses `torchmetrics.detection.MeanAveragePrecision` for a clean comparison to the YOLOv8 source baseline in `source_domain_results.csv`. Predictions are post-processed via `DetrImageProcessor.post_process_object_detection` at `threshold=0.05` (low to maximise recall in the mAP curve).

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

# Reload best for eval (in case training continued past it).
if os.path.isfile(STABLE):
    model = _build_model(STABLE).to(device).eval()
elif os.path.isfile(BEST_PT):
    model = _build_model(BEST_PT).to(device).eval()
else:
    print("NOTE: no checkpoint found; using current in-memory model for eval.")
    model = model.to(device).eval()

test_loader = DataLoader(
    test_ds, batch_size=MICRO_BATCH, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True,
)

metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
t_eval = time.time()
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
        target_sizes = torch.stack([
            torch.tensor([int(t["orig_size"][0]), int(t["orig_size"][1])])
            for t in batch["labels"]
        ]).to(device)
        results = processor.post_process_object_detection(
            outputs, threshold=0.05, target_sizes=target_sizes,
        )
        preds = [
            {"boxes": r["boxes"].cpu(), "scores": r["scores"].cpu(),
             "labels": r["labels"].cpu()}
            for r in results
        ]
        # Ground truth: convert from (cx, cy, w, h) normalized to (x1,y1,x2,y2) absolute.
        gts = []
        for t in batch["labels"]:
            h, w = int(t["orig_size"][0]), int(t["orig_size"][1])
            boxes_cxcywh = t["boxes"]
            x1 = (boxes_cxcywh[:, 0] - boxes_cxcywh[:, 2] / 2.0) * w
            y1 = (boxes_cxcywh[:, 1] - boxes_cxcywh[:, 3] / 2.0) * h
            x2 = (boxes_cxcywh[:, 0] + boxes_cxcywh[:, 2] / 2.0) * w
            y2 = (boxes_cxcywh[:, 1] + boxes_cxcywh[:, 3] / 2.0) * h
            xyxy = torch.stack([x1, y1, x2, y2], dim=-1).cpu()
            gts.append({"boxes": xyxy, "labels": t["class_labels"].cpu()})
        metric.update(preds, gts)

m = metric.compute()
mAP_50    = float(m["map_50"])
mAP_50_95 = float(m["map"])
print(f"REPORT eval | mAP@.5 = {mAP_50:0.4f} | mAP@.5:.95 = {mAP_50_95:0.4f} "
      f"| elapsed_s = {time.time()-t_eval:0.0f}")

## 9.7 — Write metrics CSV for the report

Single-row summary at `<DRIVE>/reports/tables/source_domain_results_detr.csv` with the same schema as `source_domain_results.csv` from `05_train_source.ipynb`. T-01 in the gaps tracker is fed by both files concatenated.

In [ ]:
import csv, os
TABLES = os.path.join(DRIVE, "reports", "tables")
os.makedirs(TABLES, exist_ok=True)
out_csv = os.path.join(TABLES, "source_domain_results_detr.csv")

row = {
    "model": "detr-r50",
    "stage": "source_baseline",
    "n_train": len(train_ds),
    "n_val": len(val_ds),
    "n_test": len(test_ds),
    "epochs": EPOCHS,
    "imgsz": "DETR-default-800",
    "micro_batch": MICRO_BATCH,
    "grad_accum": ACCUM,
    "lr_head": LR_HEAD,
    "lr_backbone": LR_BACKBONE,
    "map_50": round(mAP_50, 4),
    "map_50_95": round(mAP_50_95, 4),
    "weights": STABLE,
    "run_dir": RUN_DIR,
}
with open(out_csv, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(row.keys()))
    w.writeheader()
    w.writerow(row)
print("REPORT step = WRITE_CSV | path =", out_csv)
for k, v in row.items():
    print(f"  {k:>14s} = {v}")